# 1.流式调用
Stream()方法允许模型在处理请求时，实时返回部分结果，而不是等待模型处理完成后再返回所有结果。

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

In [ ]:
for chunk in model.stream("帮我解释一下什么是rag"):
    # 参数详细 text是当前的响应文本，end是不换行，flush=True是立即打印
    print(chunk.text,end="",flush=True)

RAG（Retrieval-Augmented Generation，检索增强生成）是一种让大语言模型“先查资料，再回答”的技术方案。

可以把它理解成：**让 AI 参加开卷考试**。  
模型本身有训练时学到的知识，但可能过时、缺失、记不准，甚至会“编”。RAG 的思路是：回答前先从外部知识库中检索相关内容，再把这些内容作为参考交给模型生成答案。

### RAG 的基本流程

通常分两阶段：

**1. 离线建库**
- 把文档、网页、手册、FAQ、数据库记录等整理成文本
- 切成小段（chunk）
- 用嵌入模型把每段转成向量
- 存入向量数据库，建立可检索的知识库

**2. 在线问答**
- 用户提问
- 把问题也转成向量
- 在知识库中检索最相关的若干片段
- 把这些片段和问题一起交给大模型
- 大模型基于检索到的内容组织答案

简化成一句话：

> 用户问题 → 检索相关资料 → 资料 + 问题一起给模型 → 生成回答

### 为什么需要 RAG？

大模型有几个常见问题：
- **知识可能过时**：训练数据有截止时间
- **不知道企业私有数据**：比如公司制度、产品文档、客户资料
- **容易幻觉**：不知道时也可能编一个看似合理的答案
- **难以追溯来源**：答案为什么这么说得不清楚

RAG 可以：
- 接入最新资料和私有知识
- 降低幻觉
- 让答案附带出处
- 不重新训练模型也能更新知识
- 成本通常比微调低，更新更灵活

### 一个例子

假设你问公司内部助手：

> “今年年假怎么休？”

如果只靠大模型，它可能给一个通用答案，甚至编造公司规则。  
用 RAG 的话，系统会先检索公司《员工手册》中关于年假的章节，再把相关段落交给模型，最后回答：

> “根据《员工手册》第 3.2 节，入职满一年享有 5 天年假……”

并附上文档来源。

### RAG 和微调的区别

- **RAG**：侧重“查资料”，适合知识频繁更新、需要引用来源、企业私有文档问答。
- **微调**：侧重“教模型某种风格、格式或能力”，适合固定任务、语气风格、领域能力提升。
- 实际项目里二者可以结合。

### RAG 的常见难点

RAG 不是“接个向量库就完事”，效果很依赖：
- 文档切分是否合理
- 检索是否准确
- 是否需要重排序 reran

区别
- invoke()：同步调用，返回值为一个字符串，在模型输出完成后一次性获取响应
- stream()：异步调用，返回值为一个流，流中的每个元素都是一个字符串，实时的返回响应片段。调用后，返回一个迭代器(iterator),可以通过循环来实时处理每一个新生成的chunk内容

Stream()的优势
- 响应速度快
- 交互体验更加流畅
- 可以实时处理模型输出，避免等待模型输出完成

# 2 批量调用

batch()方法允许一次性发送一组请求（包含多条独立的请求），模型会在后台并行处理，然后返回所有结果的列表

与逐个顺序调用（invoke()）不同，可以大大减少网络的往返开销和等待时间。显著提升性能、降低成本

适用场景：文档摘要、批量翻译、批量分类等

## 2.1一次性获取接收的响应
batch()特点是等待所有请求处理完毕，按照原始输入顺序返回结果列表

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL  
)

In [2]:
messages = [
    "2+2等于多少？",
    "3+3等于多少？",
    "4+4等于多少？",
]

responses = model.batch(messages)

for response in responses:
    print(response.text)

2+2等于4。
6
8


# 2.2 按完成顺序接收响应结果

当模型输入列表很大或者单个模型调用耗时差异显著时候
batch_as_completed()方法允许应用在收到第一个结果后立即返回响应，而不会等待批次内所有任务完成才响应，即batch_as_completed()每个请求完成后立yield结果（注意：结果可能会出现乱序）
但是美国返回的响应都被放在一个元组中，元组的第一个元素是模型调用的索引，第二个元素是模型调用的结果。

In [8]:
message = [
    "一句话概括rag",
    "一句话概括agent",
    "一句话概括langchain"
]

responses = model.batch_as_completed(message)

for response in responses:
    print(response)

(0, AIMessage(content='RAG（检索增强生成）是让大模型先从外部知识库检索相关信息，再基于检索结果生成回答，从而提升准确性和时效性的方法。', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文一句话概括RAG。需要简洁。RAG = 检索增强生成：先检索外部知识，再将相关内容注入生成模型以提升回答准确性和时效性。一句话。注意“一句话概括rag”。直接回答。'}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 33, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 48, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 33}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '7d127828-02de-41ad-9a1f-11124749680a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1c576-0fee-716f-b34d-e87abb6e9e7e-0', tool_calls=[], invalid_tool

## 2.3 性能的对比

In [9]:
messages = [
    "2+2等于多少？",
    "3+3等于多少？",
    "4+4等于多少？",
]

import time

start_time = time.time()
responses = model.batch(messages)
end_time = time.time()

for response in responses:
    print(response.text)

print(f"Batch processing time: {end_time - start_time} seconds")

2 + 2 = 4
6
等于 8。
Batch processing time: 0.637967586517334 seconds


In [ ]:
messages = [
    "2+2等于多少？",
    "3+3等于多少？",
    "4+4等于多少？",
]

import time

start_time = time.time()
for ins in messages:
    response = model.invoke(ins)
    print(response.text)
end_time = time.time()


print(f"Invoke processing time: {end_time - start_time} seconds")

2+2等于4。
6
4+4=8
Batch processing time: 2.408425807952881 seconds
